# Tanya — RAG build and evaluation
PE6201 · End-of-Course Project · Individual problem statement

Adapted from the **A1 Part 2** retrieval-system starter, pointed at Tanya's own corpus
(`data/`) and answer key (`eval_questions.json`) instead of the placeholder houseplant
domain. Also borrows Part 3's two-level (L1/L2) harness for the success metrics defined in
Section 7 of the problem statement.

**What this notebook does, mapped to the problem statement:**

| Section | What happens here |
|---|---|
| 6 · Data | Loads the 18-document corpus, tagged by division |
| 9 · Smallest first version | The single fixed slice: one doc → chunk → embed → top-3 → cited answer |
| 7 · Success metric & evaluation | L1 context-recall (code-checkable) + L2 faithfulness (judge model), against the baseline |
| 8 · Risks | The "break it" section tests cross-division mixing without the per-division filter |

**If a cell does not work**, read the error and try to fix it before you clear it — the same
rule as the A1 starters. Generation cells are wrapped so the notebook still runs
retrieval-only if the OpenRouter/`openai` setup is broken in your environment (this was true
on the machine this draft was built on: `import openai` failed on a local `aiohttp` version
mismatch, and `sentence_transformers`/`torch` failed to load a DLL — both are environment
issues, not the notebook's logic, and are why the TF-IDF fallback and the
`GENERATION_AVAILABLE` flag below exist).

### Setup

In [1]:
%pip -q install openai scikit-learn sentence-transformers torch --no-cache-dir

Note: you may need to restart the kernel to use updated packages.


### Connect to OpenRouter
Same pattern as the A1 notebooks: generation goes through OpenRouter with your key;
embeddings run locally and free where possible. This cell is wrapped in `try/except` so a
broken `openai` install (as seen locally while drafting this) does not stop the rest of the
notebook — `GENERATION_AVAILABLE` tells every later cell whether real answers are possible.

In [2]:
import os, getpass, time, json, re
from pathlib import Path
import numpy as np
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('Secret_key')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)

# TRY THIS (knob): any model your key can reach.
#   'openai/gpt-4o-mini' (default)  ·  'anthropic/claude-3.5-haiku'
#   'qwen/qwen-2.5-7b-instruct'     ·  'deepseek/deepseek-chat'
GEN_MODEL   = 'openai/gpt-4o-mini'
JUDGE_MODEL = 'google/gemini-2.5-flash'  # a DIFFERENT model family from the answerer: a model
                                         # grading its own output is grading its own homework.
                                         # Matches eval/harness.py and the problem statement.

TOKENS = {'in': 0, 'out': 0}

def generate(prompt: str, system: str = None, model: str = None, max_new_tokens: int = 400,
             temperature: float = 0.0) -> str:
    """temperature 0 = deterministic. Essential when comparing two approaches:
       otherwise you cannot tell a real difference from random variation."""
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=model or GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return (r.choices[0].message.content or '').strip()

print('connected · model:', GEN_MODEL, '·', generate('Reply with exactly the word: ready'))
GENERATION_AVAILABLE = True


connected · model: openai/gpt-4o-mini · ready


### Check remaining credit — free, not billed

In [3]:
import urllib.request, json as _json

def credit_check():
    if not GENERATION_AVAILABLE:
        print('skipped — generation unavailable')
        return
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.2569 of $10.00  ->  $9.7431 remaining


---
## 1 · Load Tanya's documents (Section 6)

18 documents, 6 per division, under `data/<division>/`. The id-to-path-to-division mapping
below is the same one `check_my_data.py` validates against `eval_questions.json` — keep them
in sync if you add documents.

In [4]:
DATA_DIR = Path('data')   # run this notebook from the tanya-rag-project root

DOC_ID_TO_PATH = {
    "fnb-01": "fnb/packaging_change_sop.md",       "fnb-02": "fnb/client_contract_summary.md",
    "fnb-03": "fnb/quality_control_checklist.md",  "fnb-04": "fnb/maintenance_log.md",
    "fnb-05": "fnb/hygiene_sop.md",                "fnb-06": "fnb/financial_summary_q3.md",
    "cnc-01": "cnc/tooling_spec_bottle_cap.md",    "cnc-02": "cnc/capacity_planning_report.md",
    "cnc-03": "cnc/machine_calibration_log.md",    "cnc-04": "cnc/material_procurement_policy.md",
    "cnc-05": "cnc/safety_sop.md",                 "cnc-06": "cnc/client_order_backlog.md",
    "jwl-01": "jewellery/finance_intercompany_loan_report.md",
    "jwl-02": "jewellery/gold_pricing_policy.md",
    "jwl-03": "jewellery/inventory_audit_sop.md",
    "jwl-04": "jewellery/quality_certification_sop.md",
    "jwl-05": "jewellery/retail_returns_policy.md",
    "jwl-06": "jewellery/client_quote_template.md",
}
DIVISION_NAME = {'fnb': 'fnb', 'cnc': 'cnc', 'jwl': 'jewellery'}

DOCS = []
for doc_id, rel_path in DOC_ID_TO_PATH.items():
    text = (DATA_DIR / rel_path).read_text(encoding='utf-8')
    division = DIVISION_NAME[doc_id.split('-')[0]]
    DOCS.append({'doc_id': doc_id, 'division': division, 'path': rel_path, 'text': text})

print(f'{len(DOCS)} documents loaded across {len(set(d["division"] for d in DOCS))} divisions')
for d in DOCS[:3]:
    print(f'  {d["doc_id"]:<8} {d["division"]:<10} {len(d["text"].split())} words  {d["path"]}')

18 documents loaded across 3 divisions
  fnb-01   fnb        270 words  fnb/packaging_change_sop.md
  fnb-02   fnb        203 words  fnb/client_contract_summary.md
  fnb-03   fnb        181 words  fnb/quality_control_checklist.md


---
## 2 · Chunking
Same knobs as the A1 Part 2 starter. Documents here are short (150-300 words), so start with
a moderate chunk size — you can compare against a whole-document chunk later, the way Part 2's
`test_chunk_size` scratch cell does.

In [5]:
CHUNK_WORDS = 200     # TRY THIS (knob): try 25 (tight) or 300 (whole doc = one chunk)
OVERLAP     = 50      # round 7: measured best recall-per-token tradeoff (results/chunk_sweep.md),
                      # adopted in webapp/rag_core.py too -- was 60/15

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words, out, i = text.split(), [], 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = []   # each: (chunk_text, doc_id, division)
for d in DOCS:
    for c in chunk(d['text']):
        CHUNKS.append((c, d['doc_id'], d['division']))

print(f'{len(DOCS)} docs -> {len(CHUNKS)} chunks (CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP})')

18 docs -> 21 chunks (CHUNK_WORDS=200, OVERLAP=50)


---
## 3 · Embed and retrieve
Local embeddings if `sentence-transformers` loads; TF-IDF fallback otherwise (same pattern as
Part 2 — say in your writeup which one you actually ran on, since it changes which failures
show up). `retrieve()` takes an optional `divisions` filter: passing it in is the "own the
per-division retrieval filtering" mechanism Section 5 of the problem statement argues for;
leaving it out is the naive baseline you compare against in the "break it" section.

In [6]:
# round 8: embed the document TITLE prefixed to each chunk, not the chunk alone --
# measured in results/retrieval_recall.md (titled_k5 vs naive_k5): division recall
# 91% -> 100% at THIS 200-word chunk size, at zero extra generation tokens, since only
# the embedding input changes -- retrieve() below still returns the plain chunk text.
# It does NOT fix exact-document misses (avg doc recall unchanged at 88% either way;
# CD1 below still misses cnc-01). Adopted in webapp/rag_core.py's embed_text(); mirrored
# here so this notebook measures the same pipeline the app runs, not a stale one.
TITLES = {d['doc_id']: next((ln[2:].strip() for ln in d['text'].splitlines() if ln.startswith('# ')), d['doc_id'])
          for d in DOCS}

USING = None
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    def embed(texts):
        return np.asarray(_m.encode(list(texts), normalize_embeddings=True))
    USING = 'local embeddings (all-MiniLM-L6-v2) — matches MEANING'
except Exception as e:
    print('embedder unavailable ->', str(e)[:150])
    from sklearn.feature_extraction.text import TfidfVectorizer
    _v = TfidfVectorizer().fit([f'{TITLES[doc_id]}. {c}' for c, doc_id, _ in CHUNKS])
    def embed(texts):
        m = _v.transform(list(texts)).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.where(n == 0, 1, n)
    USING = 'TF-IDF fallback — matches WORDS, not meaning'

print('using:', USING)
MATRIX = embed([f'{TITLES[doc_id]}. {c}' for c, doc_id, _ in CHUNKS])

TOP_K = 3   # TRY THIS (knob): the problem statement's smallest-first-version uses top-3

def retrieve(question, k=TOP_K, divisions=None):
    '''divisions: None = naive (no filter); or a list like ['fnb','cnc'] to filter first,
    matching the per-division tagging Section 8 relies on to prevent cross-division mixing.'''
    q = embed([question])[0]
    scores = MATRIX @ q
    if divisions is not None:
        mask = np.array([1.0 if div in divisions else -np.inf for _, _, div in CHUNKS])
        scores = scores + mask
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], CHUNKS[i][2], float(scores[i])) for i in order]

for text, doc_id, division, score in retrieve('What packaging spec changes require new CNC tooling?'):
    print(f'  {score:.3f}  [{doc_id} · {division}]  {text[:70]}...')

c:\Users\Trixie\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
c:\Users\Trixie\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Trixie\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
c:\Users\Trixie\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45

using: local embeddings (all-MiniLM-L6-v2) — matches MEANING
  0.668  [fnb-01 · fnb]  # SOP-FNB-014 — Packaging Specification Change Procedure **Division:**...
  0.620  [fnb-01 · fnb]  pilot run of 500 units is completed and inspected against QC-FNB-002 b...
  0.539  [cnc-04 · cnc]  # POL-CNC-005 — Raw Material Procurement Policy **Division:** CNC Prec...


---
## 4 · Grounded answer, with citation
Same discipline as Part 2: answer ONLY from retrieved text, decline explicitly when it is not
there, and cite the chunk relied on so a human (or the L2 judge) can check faithfulness
without re-reading the whole corpus.

In [7]:
GROUNDED = (
    "Answer using ONLY the numbered notes provided below. Each note reads "
    "\"[N] (doc_id) text\" -- the id you must cite is whatever text appears in the "
    "parentheses right after the bracket number, copied exactly, not shortened or "
    "reformatted. It is NEVER the bracket number N itself; N is only a position marker "
    "for your own reading, not an id. If the notes answer the question, even partially, "
    "use them and answer with what they support. Only if the notes truly contain nothing "
    "relevant, reply exactly: The documents do not say. Do not use any other knowledge. "
    "Finish every reply with exactly one line: 'Cited: ' followed by the real id(s) you "
    "used, comma-separated. This is a FORMAT example only, with ids that do not exist in "
    "this corpus -- never reuse them, always substitute the real id(s) from the notes "
    "above: Cited: abc-01, xyz-02"
)

def answer(question, k=TOP_K, divisions=None, show=False):
    hits = retrieve(question, k, divisions=divisions)
    context = '\n\n'.join(f'[{i+1}] ({doc_id}) {t}' for i, (t, doc_id, div, s) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {question}', system=GROUNDED)
    if show:
        print(f'Q: {question}\nA: {out}\n')
    return out, hits

_ = answer('What packaging spec changes require new CNC tooling?', show=True)

Q: What packaging spec changes require new CNC tooling?
A: Any change to cap diameter, thread pitch, or bottle neck finish requires new tooling from the CNC division — the existing filling-line tooling cannot be re-machined on-site and must be replaced via a CNC work order. Cited: fnb-01



---
## 5 · Load the answer key (Section 7)
`eval_questions.json` was written by hand *before* any of the above ran — 20 scored questions
(cross-division and single-division) plus 3 out-of-scope questions to test abstention. This is
the ground truth `check_my_data.py` already validated.

In [8]:
EVAL = json.loads(Path('eval_questions.json').read_text(encoding='utf-8'))['questions']
SCORED = [q for q in EVAL if q['kind'] != 'out_of_scope']
OOS    = [q for q in EVAL if q['kind'] == 'out_of_scope']
CROSS  = [q for q in SCORED if q['kind'] == 'cross_division']

# The hand-written answer key: the facts an answer MUST contain to be correct, fixed before
# anything was run. Faithfulness alone can be won by quoting a chunk back, so Section 7 below
# scores correctness against these as well.
KEY_FACTS = json.loads(Path('eval_key_facts.json').read_text(encoding='utf-8'))['facts']

print(f'{len(SCORED)} scored questions ({len(CROSS)} cross-division, '
      f'{len(SCORED)-len(CROSS)} single-division) · {len(OOS)} out-of-scope')
print(f'{len(KEY_FACTS)} questions have a hand-written key-fact list')

20 scored questions (11 cross-division, 9 single-division) · 3 out-of-scope


---
## 6 · L1 — context recall (code-checkable, no model call needed)
Section 7's retrieval metric: *did Tanya retrieve chunks from every division a cross-division
query touches, not just the first one found?* This is a pure code check against `MATRIX`, so
it runs even with `GENERATION_AVAILABLE = False`.

In [9]:
def context_recall_check(q, k=TOP_K):
    hits = retrieve(q['question'], k)   # naive retrieval — no division filter
    retrieved_divisions = {div for _, _, div, _ in hits}
    retrieved_doc_ids = {doc_id for _, doc_id, _, _ in hits}
    expected_divisions = set(q['expected_divisions'])
    division_recall = expected_divisions <= retrieved_divisions
    doc_overlap = len(retrieved_doc_ids & set(q['source_doc_ids'])) / max(1, len(q['source_doc_ids']))
    return division_recall, doc_overlap, hits

rows = []
for q in SCORED:
    division_recall, doc_overlap, hits = context_recall_check(q)
    rows.append((q['id'], q['kind'], division_recall, doc_overlap))
    flag = 'OK' if division_recall else 'MISS'
    print(f'{q["id"]:<6} {q["kind"]:<16} division_recall={flag:<4} doc_overlap={doc_overlap:.0%}')

cross_recall = sum(r[2] for r in rows if r[1] == 'cross_division') / len(CROSS)
overall_recall = sum(r[2] for r in rows) / len(rows)
print(f'\ncontext recall on cross-division cases: {cross_recall:.0%}  (target >=80%, Section 7)')
print(f'context recall overall: {overall_recall:.0%}')

CD1    cross_division   division_recall=OK   doc_overlap=50%
CD2    cross_division   division_recall=OK   doc_overlap=50%
CD3    cross_division   division_recall=OK   doc_overlap=67%
CD4    cross_division   division_recall=MISS doc_overlap=50%
CD5    cross_division   division_recall=OK   doc_overlap=100%
CD6    cross_division   division_recall=OK   doc_overlap=50%
CD7    cross_division   division_recall=OK   doc_overlap=100%
CD8    cross_division   division_recall=MISS doc_overlap=50%
CD9    cross_division   division_recall=MISS doc_overlap=50%
S9     single_division  division_recall=OK   doc_overlap=100%
CD11   cross_division   division_recall=MISS doc_overlap=50%
CD12   cross_division   division_recall=OK   doc_overlap=0%
S1     single_division  division_recall=OK   doc_overlap=100%
S2     single_division  division_recall=OK   doc_overlap=100%
S3     single_division  division_recall=OK   doc_overlap=100%
S4     single_division  division_recall=OK   doc_overlap=50%
S5     single_divis

---
## 7 · L2 — faithfulness, and abstention on out-of-scope questions
Needs `GENERATION_AVAILABLE`. If it is `False` in your environment, fix the OpenRouter setup
(or run this notebook in Colab, like your A1 starters) before trusting these numbers — the L1
section above is still valid evidence on its own in the meantime.

In [10]:
JUDGE_RUBRIC = '''You are grading whether an answer from Tanya (a RAG assistant over TGMOK Holdings' internal documents) is FAITHFUL to the retrieved notes it was given.

Given the QUESTION, the NOTES Tanya saw, and Tanya's ANSWER, decide:
Is every factual claim in the ANSWER actually supported by the NOTES? An answer that adds a
plausible-sounding fact not present in the NOTES is UNFAITHFUL, even if the fact happens to be
correct in the wider fictional scenario. An answer that correctly declines because the NOTES do
not cover the question is FAITHFUL.

Reply with exactly one word: FAITHFUL or UNFAITHFUL.'''

def judge_l2(question, notes_text, answer_text):
    if not GENERATION_AVAILABLE:
        return None
    verdict = generate(
        f'QUESTION: {question}\n\nNOTES:\n{notes_text}\n\nANSWER: {answer_text}',
        system=JUDGE_RUBRIC, model=JUDGE_MODEL, max_new_tokens=20)
    return verdict.strip().upper().startswith('FAITHFUL')

def norm(s):
    return (s or '').lower().replace(',', '')

def key_fact_check(answer, groups):
    """Code check, no model: every group needs at least one of its alternatives present.
    This is ANSWER CORRECTNESS -- did it say the right thing -- as opposed to faithfulness,
    which only asks whether what it said was supported by the notes it was given. A model can
    win faithfulness by quoting a chunk back; it cannot win this that way."""
    a = norm(answer)
    missing = [g for g in groups if not any(norm(alt) in a for alt in g)]
    return (len(groups) > 0 and not missing), missing

rows = []
for q in SCORED:
    ans, hits = answer(q['question'])
    notes_text = '\n'.join(f'({doc_id}) {t}' for t, doc_id, div, s in hits)
    faithful = judge_l2(q['question'], notes_text, ans)
    groups = KEY_FACTS.get(q['id'], {}).get('groups', [])
    correct, missing = key_fact_check(ans, groups)
    abstained = 'do not say' in ans.lower()
    rows.append({'id': q['id'], 'faithful': faithful, 'correct': correct,
                 'missing': missing, 'abstained': abstained})
    flag = 'ABSTAINED' if abstained else ('ok' if correct else 'MISSING ' + str(missing))
    print(f'{q["id"]:<6} faithful={str(faithful):<5} correct={str(correct):<5} {flag}')

abstain_results = []
for q in OOS:
    ans, hits = answer(q['question'])
    abstained = 'do not say' in ans.lower() or 'not covered' in ans.lower() or 'does not' in ans.lower()
    abstain_results.append((q['id'], abstained))
    print(f'{q["id"]:<8} abstained={abstained}   {ans[:90]}')

if GENERATION_AVAILABLE:
    judged = [r for r in rows if r['faithful'] is not None]
    faith_rate   = sum(1 for r in judged if r['faithful']) / max(1, len(judged))
    correct_rate = sum(1 for r in rows if r['correct']) / max(1, len(rows))
    abstain_rate = sum(1 for _, a in abstain_results if a) / max(1, len(abstain_results))

    # Abstention as the TWO numbers Section 7 asks for: how often it declines, and whether the
    # questions it declined were the ones it would have got wrong anyway.
    declined = [r for r in rows if r['abstained']]
    answered = [r for r in rows if not r['abstained']]
    n_wrong  = sum(1 for r in answered if not r['correct'])

    print(f'\nfaithfulness:       {faith_rate:.0%}  (target >=85%, Section 7)')
    print(f'answer correctness:  {correct_rate:.0%}  (key facts present -- the check faithfulness cannot fake)')
    print(f'abstention on out-of-scope questions: {abstain_rate:.0%}  (should be high)')
    print(f'\nAbstention, two numbers (Section 7):')
    print(f'  declined an answerable question: {len(declined)/max(1,len(rows)):.0%} ({len(declined)}/{len(rows)})')
    print(f'  of the ones it DID answer, wrong: {n_wrong/max(1,len(answered)):.0%} ({n_wrong}/{len(answered)})')
    print('  -> declining only pays if those questions would have been wrong; compare the two rates.')


CD1    faithful=True   The packaging change that forced F&B to order new CNC tooling was a request from Client Au
CD2    faithful=False   Yes, the CNC tooling for the Aurora packaging change was installed immediately. It was del
CD3    faithful=True   The documents do not say.  
Cited:
CD4    faithful=True   External CNC client order C-4471 was delayed by 4 days to accommodate the internal work or
CD5    faithful=True   The second capping head (FL2-CAP-02) was installed ahead of the Aurora rush order on 2026-
CD6    faithful=True   The specific external CNC order that was pushed back due to the F&B rush order is Order #C
CD7    faithful=True   CNC Production Planning recommended that F&B Client Services flag rush-order possibilities
CD8    faithful=True   F&B funded its second capping head through an intercompany working-capital facility extend
CD9    faithful=True   CNC maintains a rolling 2-week buffer of MS-07 tool steel specifically to absorb expedited
S9     faithful=True   A piec

---
## 8 · Baseline (Section 7 requires one)
The same questions answered with **keyword (TF-IDF) retrieval** instead of the app's semantic
embeddings — this is what the faithfulness/context-recall numbers above are actually being
compared against. A no-retrieval bare-model baseline (this cell's original version) is rigged:
gpt-4o-mini has never seen these fictional documents, so it would score near zero regardless of
how good or bad the real retrieval is, and that comparison teaches nothing. A non-AI keyword
search over the exact same chunks is the fair comparison — if the real system does no better
than simple keyword matching, semantic embeddings haven't earned their complexity either.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Keyword (TF-IDF) retrieval over the exact same CHUNKS the real system uses, so this is a fair
# non-AI baseline (per reviewer feedback) rather than the no-retrieval version above.
_tfidf_vec = TfidfVectorizer().fit([c for c, _, _ in CHUNKS])
_tfidf_matrix = _tfidf_vec.transform([c for c, _, _ in CHUNKS]).toarray()
_tfidf_norms = np.linalg.norm(_tfidf_matrix, axis=1, keepdims=True)
_tfidf_matrix = _tfidf_matrix / np.where(_tfidf_norms == 0, 1, _tfidf_norms)

def retrieve_tfidf(question, k=TOP_K):
    qv = _tfidf_vec.transform([question]).toarray()[0]
    qn = np.linalg.norm(qv)
    qv = qv / qn if qn else qv
    scores = _tfidf_matrix @ qv
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], CHUNKS[i][2], float(scores[i])) for i in order]

baseline_rows = []
for q in SCORED[:5]:   # a handful is enough to illustrate; extend to all 20 for the real run
    hits = retrieve_tfidf(q['question'])
    context = '\n\n'.join(f'[{i+1}] ({doc_id}) {t}' for i, (t, doc_id, div, s) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {q["question"]}', system=GROUNDED)
    baseline_rows.append((q['id'], out))
    print(f'{q["id"]:<6} (TF-IDF retrieval) {out[:100]}')

CD1    (no retrieval) The shift to sustainable packaging, such as biodegradable or recyclable materials, often necessitate
CD2    (no retrieval) I don't have specific information on the installation of CNC tooling for the Aurora packaging change
CD3    (no retrieval) The time from packaging-change request to tooling delivery can vary, but it typically takes several 
CD4    (no retrieval) I'm sorry, but I don't have access to specific order details or real-time data regarding external CN
CD5    (no retrieval) The second capping head installation that followed CNC's June 2026 capacity reprioritisation was ins


---
## 9 · Break it (Section 8) — cross-division mixing without the filter
Retrieve **without** the division filter on a cross-division question and see whether a
wrong-division chunk outranks a correct one — this is the concrete failure mode Section 8's
mitigation (tag every chunk with its division, filter at query time) is meant to prevent.
Diagnose from the scores, the way the A1 Part 2 `FAILURE_DIAGNOSIS` cell does.

In [12]:
BREAK_Q = next(q for q in CROSS if q['id'] == 'CD1')   # packaging -> CNC tooling hook
print('question:', BREAK_Q['question'])
print('expected divisions:', BREAK_Q['expected_divisions'], '· expected docs:', BREAK_Q['source_doc_ids'])

print('\n-- naive retrieval (no division filter) --')
for t, doc_id, div, s in retrieve(BREAK_Q['question'], k=5):
    print(f'  {s:.3f}  [{doc_id} · {div}]  {t[:70]}...')

print('\n-- filtered retrieval (divisions=expected_divisions) --')
for t, doc_id, div, s in retrieve(BREAK_Q['question'], k=5, divisions=BREAK_Q['expected_divisions']):
    print(f'  {s:.3f}  [{doc_id} · {div}]  {t[:70]}...')

FAILURE_DIAGNOSIS = """Question: CD1, "What packaging change forced F&B to order new CNC tooling, and what was
delivered?" Expected documents: fnb-01 (the change) and cnc-01 (what was delivered).

What went wrong: cnc-01 is still NOT retrieved by naive top-5 -- it scores 0.432 against a
0.477 cutoff for 5th place, ranked #8 of 21 chunks. The CNC chunks that WERE retrieved instead
are cnc-04 (Raw Material Procurement Policy), cnc-02 (Capacity Planning Report), and cnc-06
(Client Order Backlog) -- three OTHER CNC documents, none of which is the actual answer. So a
division-level check still passes (a CNC chunk is present) while the needed DOCUMENT is still
absent -- the same "right division, wrong document" failure mode as before round 7, just with
a narrower score gap (0.045 now, was 0.086 at the old 60/15 config).

The division filter still does not help: all five naive results are already F&B or CNC, so
filtering changes nothing.

Mechanism, revised: cnc-01 is 198 words, so at the now-adopted 200-word chunk size it is ONE
whole-document chunk -- there is no longer a separate, metadata-heavy "worst chunk" diluting
a cleaner one, the way there was at 60 words. The whole document, header and substance
together, still scores below cutoff. The better-supported mechanism now: the query's generic
phrasing ("packaging change", "new CNC tooling", "delivered") shares more surface vocabulary
with OTHER CNC documents about orders, capacity, and procurement than with cnc-01's own
specific engineering language ("capping-head assembly", "26mm neck/cap diameter", "thread
pitch"). The right document answers in more specific technical vocabulary than the question
asks in, and that specificity is itself what pushes it down the ranking.

What actually "fixes" it now is different from before, and worth being precise about because
it reverses the earlier conclusion: title-prefixed embedding (adopted as the shipped default
in round 8, for its AGGREGATE benefit across the full eval set) does NOT fix this specific
case -- cnc-01 still ranks #8 with titles on. Testing PLAIN (untitled) embedding at the SAME
200-word chunking instead, cnc-01 ranks #6 (score 0.466) -- closer to top-5, though still
short of it. At the old 60-word config, titling was the thing that fixed this exact question;
at 200 words, titling is neutral-to-slightly-worse for this one question specifically, even
though it is measurably better in aggregate (see cell 26). That reversal is the real lesson,
not either specific ranking: a knob's effect on one failing case does not transfer across a
different config change, and a fix should be judged on the full evaluation set, not the case
that motivated looking for it.
"""
print(FAILURE_DIAGNOSIS)


question: What packaging change forced F&B to order new CNC tooling, and what was delivered?
expected divisions: ['fnb', 'cnc'] · expected docs: ['fnb-01', 'cnc-01']

-- naive retrieval (no division filter) --
  0.660  [fnb-01 · fnb]  # SOP-FNB-014 — Packaging Specification Change Procedure **Division:**...
  0.603  [fnb-01 · fnb]  pilot run of 500 units is completed and inspected against QC-FNB-002 b...
  0.502  [cnc-04 · cnc]  # POL-CNC-005 — Raw Material Procurement Policy **Division:** CNC Prec...
  0.492  [cnc-02 · cnc]  # CNC Capacity Planning Report — June 2026 **Division:** CNC Precision...
  0.477  [cnc-06 · cnc]  # Client Order Backlog — CNC Division, June 2026 Snapshot **Division:*...

-- filtered retrieval (divisions=expected_divisions) --
  1.660  [fnb-01 · fnb]  # SOP-FNB-014 — Packaging Specification Change Procedure **Division:**...
  1.603  [fnb-01 · fnb]  pilot run of 500 units is completed and inspected against QC-FNB-002 b...
  1.502  [cnc-04 · cnc]  # POL-CNC-005 —

---
## 9.5 · After break it: does anything actually fix it?
Same move as the A1 Part 2 starter's chunk-size-sweep cell after its own failure diagnosis: having named *why* CD1 fails, try the obvious knobs on that one question and see if they work — and if a fix that helps this one case might hurt everywhere else.

In [13]:
### TRY THIS (knob), same pattern as A1 Part 2's chunk-size-sweep cell: having diagnosed
### WHY CD1 fails under the round 7-8 config (200/50, titled embedding), does anything fix
### it now? Re-run against this one question, same as before -- the answer changed.

# Knob 1: raise k. cnc-01 is now a single chunk (the whole 198-word document fits in one
# 200-word chunk), ranked #8 of 21 with titling on -- a much smaller corpus and a much
# smaller k than before (was #14 of 74), but also a much smaller corpus to begin with.
print('-- knob 1: how large does k have to get before cnc-01 shows up? --')
for k in (3, 5, 6, 8, 10, 15, 21):
    hits = retrieve(BREAK_Q['question'], k=k)
    got = any(doc_id == 'cnc-01' for _, doc_id, _, _ in hits)
    words = sum(len(t.split()) for t, _, _, _ in hits)
    print(f'  k={k:<3} words sent={words:<5} cnc-01 retrieved: {got}')

# Knob 2: titling is now the SHIPPED DEFAULT (round 8, adopted for its aggregate benefit --
# see results/retrieval_recall.md), so the open question is reversed from before: does it
# still help on THIS specific case, or was CD1's earlier "fix" specific to the old chunk
# size? Build a PLAIN (untitled) index at the same 200/50 chunking to compare against.
print('\n-- knob 2: is titling still what fixes THIS question, now that it is the default? --')
PLAIN_MATRIX = embed([c for c, _, _ in CHUNKS])

def retrieve_plain(question, k=TOP_K):
    q = embed([question])[0]
    scores = PLAIN_MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], CHUNKS[i][2], float(scores[i])) for i in order]

for text, doc_id, division, score in retrieve_plain(BREAK_Q['question'], k=5):
    print(f'  {score:.3f}  [{doc_id} \u00b7 {division}]  {text[:70]}...')
got_plain = any(doc_id == 'cnc-01' for _, doc_id, _, _ in retrieve_plain(BREAK_Q['question'], k=5))
print(f'  cnc-01 retrieved WITHOUT titles, top-5: {got_plain}  (WITH titles, top-5: False -- see diagnosis above)')

IMPROVEMENT_VERDICT = """Neither knob fixes THIS question for free, and knob 2's answer flipped from before:

Knob 1 (larger k): cnc-01 needs k=8 now, not k=14 -- but the corpus also shrank from 74 chunks
to 21, so 8/21 (38%) is actually a LARGER share of the corpus than 14/74 (19%) was. In words
actually sent, k=8 costs about 1,360 words for this question against k=5's 884 -- roughly 1.5x,
a real cost increase for a fix that only targets one question.

Knob 2 (title-prefixed chunks): this REVERSES the original finding. At 60-word chunks, titling
was what fixed CD1 (cnc-01 made top-5 with titles on). At the now-adopted 200-word chunks,
titling does NOT fix CD1 -- cnc-01 ranks #8 with titles, #6 without them; plain embedding is
now slightly CLOSER to catching this specific case. Titling is still the right default choice
-- it measures 100% division recall in aggregate across all 20 scored questions, against 91%
for plain (results/retrieval_recall.md) -- but that is an aggregate result, not evidence that
it helps any one question, including the one that originally motivated testing it. A fix's
generalization has to be checked on the full set every time a config changes, not assumed to
carry over.

What I'd actually ship, and why: title-prefixed top-5 (100% division recall in aggregate,
confirmed in results/retrieval_recall.md) plus the app's citation-content check (rag_core.
unsupported_figures), which cannot retrieve the missing document but DOES catch a wrong or
invented answer downstream if the model tries to answer from cnc-02, cnc-04, or cnc-06
instead. Retrieval and generation-time checks are complementary, not substitutes for each
other -- true before this round's config change, and still true after it.
"""
print(IMPROVEMENT_VERDICT)


-- knob 1: how large does k have to get before cnc-01 shows up? --
  k=3   words sent=502   cnc-01 retrieved: False
  k=5   words sent=884   cnc-01 retrieved: False
  k=6   words sent=1032  cnc-01 retrieved: False
  k=8   words sent=1358  cnc-01 retrieved: True
  k=10  words sent=1739  cnc-01 retrieved: True
  k=15  words sent=2522  cnc-01 retrieved: True
  k=21  words sent=3378  cnc-01 retrieved: True

-- knob 2: is titling still what fixes THIS question, now that it is the default? --
  0.653  [fnb-01 · fnb]  # SOP-FNB-014 — Packaging Specification Change Procedure **Division:**...
  0.568  [fnb-01 · fnb]  pilot run of 500 units is completed and inspected against QC-FNB-002 b...
  0.494  [cnc-02 · cnc]  # CNC Capacity Planning Report — June 2026 **Division:** CNC Precision...
  0.485  [cnc-04 · cnc]  # POL-CNC-005 — Raw Material Procurement Policy **Division:** CNC Prec...
  0.476  [cnc-06 · cnc]  # Client Order Backlog — CNC Division, June 2026 Snapshot **Division:*...
  cnc-01 retr

---
## 10 · Smallest first version (Section 9)
The exact slice named in the problem statement: one F&B document, chunked, embedded, top-3
retrieved for one fixed query, answered with citation — checked by hand against the source.

In [14]:
SMALLEST_SLICE_QUERY = "What packaging spec changes require new CNC tooling?"
ans, hits = answer(SMALLEST_SLICE_QUERY, k=3, divisions=None, show=True)

print('Manual check: does the cited chunk actually contain the fact used in the answer?')
for t, doc_id, div, s in hits:
    print(f'  [{doc_id}] {t}')

Q: What packaging spec changes require new CNC tooling?
A: Any change to cap diameter, thread pitch, or bottle neck finish requires new tooling from the CNC division — the existing filling-line tooling cannot be re-machined on-site and must be replaced via a CNC work order. Cited: fnb-01

Manual check: does the cited chunk actually contain the fact used in the answer?
  [fnb-01] # SOP-FNB-014 — Packaging Specification Change Procedure **Division:** F&B Contract Manufacturing (TGMOK Holdings) **Effective:** 2026-06-01 · **Owner:** Packaging Engineering, F&B ## Purpose Governs how a client-requested change to bottle, cap, or carton specification is evaluated, approved, and rolled out on the production line. ## Procedure 1. Client Services logs the requested change in the Packaging Change Register within 1 business day of receipt. 2. Packaging Engineering assesses whether the change is compatible with existing filling and capping tooling. **Any change to cap diameter, thread pitch, or bot

---
## 11 · Token spend and summary

In [15]:
print(f'tokens this session: {TOKENS["in"]} in / {TOKENS["out"]} out')
print(f'embedding backend: {USING}')
print(f'chunking: CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP}, TOP_K={TOP_K}')

tokens this session: 45582 in / 1470 out
embedding backend: local embeddings (all-MiniLM-L6-v2) — matches MEANING
chunking: CHUNK_WORDS=200, OVERLAP=50, TOP_K=3


---
## What to write up

- Which embedding backend actually ran (`USING`) — say so, since TF-IDF vs local embeddings
  changes which failures you'd expect to see.
- The L1 context-recall numbers from Section 6, against the >=80% cross-division target.
- The L2 faithfulness and abstention numbers from Section 7, against the >=85% target — and
  whether `GENERATION_AVAILABLE` was `True` when you measured them.
- The one diagnosed failure from Section 9 (break it): which chunk should have won, which did,
  and the mechanism.
- The baseline comparison from Section 8: did retrieval actually help versus the bare model?